## Notebook23b

In this notebook, you will be able to load in the saved state from the previous notebook and play around with the text that is generated by the model.

### Setup

In [ ]:
!mkdir -p models
!wget -q -nc -P models https://humanitiesdata.org/models/baby_llm.pth

### Model Definition

This is the same architecture from the training notebook — we need to define it so PyTorch knows what shape the weights should be.

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, context_length, dropout=0.0):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        mask = torch.triu(torch.ones(context_length, context_length, dtype=torch.bool), diagonal=1)
        self.register_buffer("mask", mask)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        att = att.masked_fill(self.mask[:T, :T], float("-inf"))
        att = self.dropout(F.softmax(att, dim=-1))
        out = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(out)


class FeedForward(nn.Module):
    def __init__(self, d_model, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model), nn.GELU(),
            nn.Linear(4 * d_model, d_model), nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, context_length, dropout=0.0):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, context_length, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = FeedForward(d_model, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x


class BabyLLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, context_length, dropout=0.0):
        super().__init__()
        self.context_length = context_length
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(context_length, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.Sequential(*[
            TransformerBlock(d_model, n_heads, context_length, dropout)
            for _ in range(n_layers)
        ])
        self.ln_final = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.token_emb.weight

    def forward(self, idx):
        B, T = idx.shape
        tok = self.token_emb(idx)
        pos = self.pos_emb(torch.arange(T, device=idx.device))
        x = self.ln_final(self.blocks(self.drop(tok + pos)))
        return self.head(x)

### Load Weights

Upload `baby_llm.pth` to your Colab files (or adjust the path below), and we load the trained weights into the model.

In [ ]:
model = BabyLLM(
    vocab_size=enc.n_vocab,
    d_model=384,
    n_heads=4,
    n_layers=6,
    context_length=CONTEXT_LENGTH,
).to(device)

model.load_state_dict(torch.load("models/baby_llm.pth", map_location=device, weights_only=True))
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"Loaded model with {n_params:,} parameters")

And here is the forward definition that we will use to generate new text.

In [ ]:
@torch.no_grad()
def generate(model, prompt, max_new_tokens, temperature, top_k):
    tokens = torch.tensor(enc.encode(prompt), dtype=torch.long, device=device).unsqueeze(0)
    for _ in range(max_new_tokens):
        context = tokens[:, -CONTEXT_LENGTH:]
        logits = model(context)[:, -1, :] / temperature
        if top_k is not None:
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, [-1]]] = float("-inf")
        tokens = torch.cat([tokens, torch.multinomial(F.softmax(logits, dim=-1), 1)], dim=1)
    return enc.decode(tokens[0].tolist())

### Generate Text

Take some time to explore the model by using the parameters below. Unless temperature is zero, the model should produce different things each time and you can test what happens when running the same values multiple iterations.

In [ ]:
prompt = "The stock market"
temperature = 0.8
max_tokens = 100
top_k = 40

print(generate(model, prompt, max_tokens, temperature, top_k))